# Extracting clock-time features

This notebook extracts hour, minute, second, time-of-day groups, and cyclical representations from timestamps. Clock time is often more useful than the full date for demand, fraud, web, and operations problems.

## Why use cyclical encoding?

Hour 23 and hour 0 are close in real life but far apart as ordinary integers. Sine and cosine place the beginning and end of a cycle next to each other while preserving its position.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid',context='notebook')

rng=np.random.default_rng(7)
events=pd.DataFrame({'event_time':pd.date_range('2024-01-01',periods=24*14,freq='h')})
events['demand']=20+8*np.sin(2*np.pi*events['event_time'].dt.hour/24)+rng.normal(0,1.5,len(events))
events.head()

## 1. Extract hour, minute, and second

Convert to datetime first. The derived fields are numeric and can be used directly by tree models or encoded/transformed for other models.

In [ ]:
events['event_time']=pd.to_datetime(events['event_time'],utc=True)
dt=events['event_time'].dt
events['hour']=dt.hour
events['minute']=dt.minute
events['second']=dt.second
events['minute_of_day']=dt.hour*60+dt.minute
events['part_of_day']=pd.cut(events['hour'],bins=[-1,5,11,16,20,23],labels=['night','morning','afternoon','evening','late_night'])
display(events.head())

## 2. Create cyclical features

For a cycle with period P, calculate sin(2πx/P) and cos(2πx/P). Both columns are needed: one alone cannot uniquely identify every position in the cycle.

In [ ]:
def add_cyclic_features(frame,column,period):
    result=frame.copy()
    result[f'{column}_sin']=np.sin(2*np.pi*result[column]/period)
    result[f'{column}_cos']=np.cos(2*np.pi*result[column]/period)
    return result

events=add_cyclic_features(events,'hour',24)
events=add_cyclic_features(events,'minute_of_day',24*60)
display(events[['event_time','hour','hour_sin','hour_cos','minute_of_day_sin','minute_of_day_cos']].head())

## 3. Visualise the daily cycle

The line plot shows demand changing through the day. The scatter plot shows that the cyclical hour representation traces a circle; this is why the pair of features can represent adjacency around midnight.

In [ ]:
hourly=events.groupby('hour',as_index=False)['demand'].mean()
fig,axes=plt.subplots(1,2,figsize=(14,4))
sns.lineplot(data=hourly,x='hour',y='demand',marker='o',color='#4C78A8',ax=axes[0]); axes[0].set_title('Average demand by hour')
sns.scatterplot(data=events,x='hour_sin',y='hour_cos',hue='hour',palette='hsv',legend=False,ax=axes[1]); axes[1].set_title('Cyclical representation of hour'); axes[1].set_aspect('equal')
plt.tight_layout(); plt.show()

## 4. Choosing a representation

Use a raw hour when tree thresholds or simple business rules are appropriate. Use part_of_day when stakeholders think in named periods. Use sine and cosine for smooth repeating patterns in linear models, neural networks, or distance-based methods. Always validate which representation performs best.

## Summary

We extracted clock-time components and represented repeating daily time with a pair of cyclical features. The same idea applies to weekday with period 7, month with period 12, or minute with period 60.